Some code used to do benchmarking of operations on TileDB-SOMA experiments/collections to assess what factors influence the speed & memory requirements

In [ ]:
import tiledbsoma.io
import tiledbsoma as soma
import scanpy as sc
import numpy as np
import scanpy as sc
import anndata as ad
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import psutil
import time
import os
import gc
import tracemalloc

print('ok')

ok


### Create experiments with sub-set of data with specific number of cells and genes from one h5ad file

In [ ]:
adata_prj = sc.read_h5ad('/wip/scds/delivery-zips/batch14/PRJEB39470/deliverables/Li_2020_ENA-PRJEB39470-anndata-annotated.h5ad')

def create_sub_expt(obs_num, var_num):
    # Sample cell and gene indices
    sampled_obs_indices = np.random.choice(len(adata_prj), size=min(obs_num, len(adata_prj)), replace=False)
    sampled_var_indices = np.random.choice(len(adata_prj.var), size=min(var_num, len(adata_prj.var)), replace=False)
    
    # Subset directly from the original anndata
    query_adata = adata_prj[sampled_obs_indices, :][:, sampled_var_indices].copy()
    
    return query_adata

for i in range(1, 51):
    obs_num = 2000
    var_num = 10000
    query_adata = create_sub_expt(obs_num, var_num)
    # print(query_adata)
    print(i, query_adata.obs_names[:2], query_adata.var_names[:2])
    
    tiledbsoma_expt_path = f'/home/sinu.paul/Projects/RIW-337/benchmark/experiments/expts_50x2000x2000/query_expt_{obs_num}x{var_num}_{i}'
    tiledbsoma.io.from_anndata(
        experiment_uri=tiledbsoma_expt_path,
        anndata=query_adata,
        measurement_name="RNA"
    )

### Collections

### Create collections

In [6]:
soma.Collection.create(uri='/home/sinu.paul/Projects/RIW-337/benchmark/collections/colln_10x2000x2000')

<Collection '/home/sinu.paul/Projects/RIW-337/benchmark/collections/colln_10x2000x2000' (open for 'w') (empty)>

### Add experiments to collections

In [ ]:
process = psutil.Process(os.getpid())

mem_before = process.memory_info().rss / 1024**2

start = time.perf_counter()

with soma.Collection.open('/home/sinu.paul/Projects/RIW-337/benchmark/collections/colln_50x2000x10000', 'w') as coll:
    for i in range(1, 51):
        expt_name = '/home/sinu.paul/Projects/RIW-337/benchmark/experiments/expts_50x2000x10000/query_expt_2000x10000_' + str(i)
        expt = soma.Experiment.open(expt_name)
        coll.set(expt_name, expt)
        print(f'Added {expt_name}')

end = time.perf_counter()

mem_after = process.memory_info().rss / 1024**2

print(f"Time: {end - start:.4f} sec")
print(f"Memory increase: {mem_after - mem_before:.2f} MB")

### Get list of collections

In [8]:
collections_path = Path("/home/sinu.paul/Projects/RIW-337/benchmark/collections")
dirs = [d for d in collections_path.iterdir() if d.is_dir()]

for i, dir in enumerate(dirs, start=1):
    collection_uri = str(dir)
    try:
        with soma.Collection.open(collection_uri) as collection:
            print(f"{i}. {dir.name}")
    except Exception:
        continue

1. colln_50x2000x2000
2. colln_50x10000x2000
3. colln_50x2000x10000
4. colln_10x2000x2000


### Get list of experiments in collections 

In [9]:
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None)
from urllib.parse import urlparse

collections_path = Path("/home/sinu.paul/Projects/RIW-337/benchmark/collections")
dirs = [d for d in collections_path.iterdir() if d.is_dir()]
experiments = []


for i, dir in enumerate(dirs, start=1):
    collection_uri = str(dir)
    try:
        with soma.Collection.open(collection_uri) as collection:
            for key in collection.keys():
                obj = collection[key]
                if isinstance(obj, soma.Experiment):
                    experiments.append(
                        {
                            "Collection Name": dir.name,
                            "Experiment": key,
                            "Experiment path": Path(urlparse(obj.uri).path)

                        }
                    )
    except Exception:
        continue

experiments = pd.DataFrame(experiments)
print(experiments.to_string(index=False))  

    Collection Name                                                                                          Experiment                                                                                     Experiment path
 colln_50x2000x2000    /home/sinu.paul/Projects/RIW-337/benchmark/experiments/expts_50x2000x2000/query_expt_2000x2000_1    /home/sinu.paul/Projects/RIW-337/benchmark/experiments/expts_50x2000x2000/query_expt_2000x2000_1
 colln_50x2000x2000   /home/sinu.paul/Projects/RIW-337/benchmark/experiments/expts_50x2000x2000/query_expt_2000x2000_10   /home/sinu.paul/Projects/RIW-337/benchmark/experiments/expts_50x2000x2000/query_expt_2000x2000_10
 colln_50x2000x2000   /home/sinu.paul/Projects/RIW-337/benchmark/experiments/expts_50x2000x2000/query_expt_2000x2000_11   /home/sinu.paul/Projects/RIW-337/benchmark/experiments/expts_50x2000x2000/query_expt_2000x2000_11
 colln_50x2000x2000   /home/sinu.paul/Projects/RIW-337/benchmark/experiments/expts_50x2000x2000/query_expt_2000x2000_12 

### Get number of cells in collections

In [54]:
def summarize_collection(collection_uri, measurement_name="RNA"):
    rows = []
    total_cells = 0

    with soma.Collection.open(collection_uri) as collection:
        for name, expt in collection.items():
            try:
                n_cells = expt.obs.count
                n_genes = expt.ms[measurement_name].var.count

                total_cells += n_cells
                rows.append((name, n_cells, n_genes))

            except Exception as e:
                rows.append((name, f"ERROR: {e}", None))

    df = pd.DataFrame(rows, columns=["experiment", "n_cells", "n_genes"])

    df.index = range(1, len(df) + 1)

    return df, total_cells

#collections = ["colln_10x2000x2000"]#, "colln_50x2000x2000", "colln_50x2000x10000", "colln_50x10000x2000"]
collections = ["colln_50x10000x2000"]
gc.collect()
process = psutil.Process(os.getpid())
print(process)
# mem_before = process.memory_info().rss / 1024**2

tracemalloc.start()
start = time.perf_counter()

print(f"\nCollection: {collections[0]}")
df, total_cells = summarize_collection(f"/home/sinu.paul/Projects/RIW-337/benchmark/collections/{collections[0]}")
# print(df)
print(f"Total cells in collection: {total_cells:,}")

end = time.perf_counter()
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

# mem_after = process.memory_info().rss / 1024**2
print(f"Peak memory increase: {peak / 1024**2:.2f} MB")

print(f"Time: {end - start:.4f} sec")
#print(f"Memory increase: {mem_after - mem_before:.2f} MB")


psutil.Process(pid=3565044, name='python', status='running')

Collection: colln_50x10000x2000
Total cells in collection: 490,000
Peak memory increase: 0.73 MB
Time: 1.3270 sec


### Querying for specific type of cells (e.g., "T cell") in the experiments

In [67]:
import anndata as ad

def query_cell_type_in_collection(collection, cell_type_to_query):
    collection_path = "/home/sinu.paul/Projects/RIW-337/benchmark/collections/" + collection
    collection = soma.Collection.open(collection_path)
    adatas = []

    for name, expt in collection.items():
        obs_df = expt.obs.read().concat().to_pandas()
        cell_type_labels = obs_df["popv_prediction_ontology_name"][(
            obs_df["popv_prediction_ontology_name"]
            .str.contains(cell_type_to_query, na=False)
        )].unique()

        print(f'Dataset {name} \n - Cell types based on query "{cell_type_to_query}": {list(cell_type_labels)}')

        if len(cell_type_labels) > 0:

            with expt.axis_query(
                measurement_name="RNA",
                obs_query=soma.AxisQuery(
                    value_filter=f"popv_prediction_ontology_name in {list(cell_type_labels)}"
                )
            ) as query:
                print(f' - n_obs, n_vars: {query.n_obs} cells, {query.n_vars} vars\n')
                adata = query.to_anndata(X_name="data")
                adata.obs["source_experiment"] = name
                adatas.append(adata)

    combined_adata = ad.concat(adatas, join="outer", label="experiment")
    return combined_adata

tracemalloc.start()
start = time.perf_counter()

collection = 'colln_50x2000x10000'
cell_type_to_query = 'T cell'
try:
    combined_adata = query_cell_type_in_collection(collection, cell_type_to_query)
    # combined_adata
except Exception as e:
    pass

end = time.perf_counter()
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"Peak memory increase: {peak / 1024**2:.2f} MB")
print(f"Time: {end - start:.4f} sec")

Dataset /home/sinu.paul/Projects/RIW-337/benchmark/experiments/expts_50x2000x10000/query_expt_2000x10000_1 
 - Cell types based on query "T cell": ['central memory CD4-positive, alpha-beta T cell', 'central memory CD8-positive, alpha-beta T cell', 'effector memory CD4-positive, alpha-beta T cell', 'effector memory CD8-positive, alpha-beta T cell, terminally differentiated', 'effector memory CD8-positive, alpha-beta T cell', 'gamma-delta T cell', 'mucosal invariant T cell', 'regulatory T cell']
 - n_obs, n_vars: 1019 cells, 10000 vars

Dataset /home/sinu.paul/Projects/RIW-337/benchmark/experiments/expts_50x2000x10000/query_expt_2000x10000_10 
 - Cell types based on query "T cell": ['effector memory CD8-positive, alpha-beta T cell', 'central memory CD8-positive, alpha-beta T cell', 'central memory CD4-positive, alpha-beta T cell', 'effector memory CD8-positive, alpha-beta T cell, terminally differentiated', 'gamma-delta T cell', 'mucosal invariant T cell', 'regulatory T cell', 'effector 

/opt/mamba/envs/tiledbsoma/lib/python3.11/site-packages/anndata/_core/anndata.py:1774: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Peak memory increase: 670.38 MB
Time: 58.8421 sec


### Querying for specific type of cells (e.g., "T cell") and nFeature_RNA > 500, in the experiments

In [75]:
def query_based_on_cell_and_gene_in_collection(collection, cell_type_to_query, gene_parameter, min_features=0):
    collection_path = "/home/sinu.paul/Projects/RIW-337/benchmark/collections/" + collection
    collection = soma.Collection.open(collection_path)
    adatas = []

    for name, expt in collection.items():
        obs_df = expt.obs.read().concat().to_pandas()
        cell_type_labels = obs_df["popv_prediction_ontology_name"][(
            obs_df["popv_prediction_ontology_name"]
            .str.contains(cell_type_to_query, na=False)
        )].unique()

        print(f'Dataset {name} \n - Cell types based on query "{cell_type_to_query}": {list(cell_type_labels)}')

        if len(cell_type_labels) > 0:

            value_filter = (
                f"popv_prediction_ontology_name in {list(cell_type_labels)} "
                f"and {gene_parameter} > {min_features}"
            )

            with expt.axis_query(
                measurement_name="RNA",
                obs_query=soma.AxisQuery(
                    value_filter=value_filter
                )
            ) as query:
                print(f' - n_obs, n_vars: {query.n_obs} cells, {query.n_vars} vars\n')
                adata = query.to_anndata(X_name="data")
                adata.obs["source_experiment"] = name
                adatas.append(adata)

    combined_adata_2 = ad.concat(adatas, join="outer", label="experiment")
    return combined_adata_2

tracemalloc.start()
start = time.perf_counter()

collection = 'colln_50x10000x2000'
cell_type_to_query = 'T cell'
gene_parameter = 'nFeature_RNA'
min_features = 500
try:
    combined_adata2 = query_based_on_cell_and_gene_in_collection(collection, cell_type_to_query, gene_parameter, min_features)
    combined_adata2
except Exception as e:
    pass

end = time.perf_counter()
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"Peak memory increase: {peak / 1024**2:.2f} MB")
print(f"Time: {end - start:.4f} sec")

Dataset /home/sinu.paul/Projects/RIW-337/benchmark/experiments/expts_50x10000x2000/query_expt_10000x2000_1 
 - Cell types based on query "T cell": ['central memory CD4-positive, alpha-beta T cell', 'regulatory T cell', 'central memory CD8-positive, alpha-beta T cell', 'gamma-delta T cell', 'effector memory CD8-positive, alpha-beta T cell, terminally differentiated', 'effector memory CD8-positive, alpha-beta T cell', 'mucosal invariant T cell', 'effector memory CD4-positive, alpha-beta T cell', 'mature NK T cell']
 - n_obs, n_vars: 4929 cells, 2000 vars

Dataset /home/sinu.paul/Projects/RIW-337/benchmark/experiments/expts_50x10000x2000/query_expt_10000x2000_10 
 - Cell types based on query "T cell": ['central memory CD4-positive, alpha-beta T cell', 'central memory CD8-positive, alpha-beta T cell', 'effector memory CD8-positive, alpha-beta T cell, terminally differentiated', 'effector memory CD8-positive, alpha-beta T cell', 'gamma-delta T cell', 'regulatory T cell', 'effector memory CD

In [76]:
combined_adata2

AnnData object with n_obs × n_vars = 48576 × 36601
    obs: 'soma_joinid', 'orig.ident', 'nFeature_RNA', 'nCount_RNA', 'percent.mt', 'percent.ribo', 'percent.hemoglobin', 'S_score', 'G2M_score', 'cell_cycle_phase', 'gene_gini', 'gene_entropy', 'mt_gini', 'mt_entropy', 'leiden', 'sample_id', 'author_cell_id', 'author_cell_type', 'author_cell_type_cell_ontology_name', 'author_cell_type_cell_ontology_id', 'author_cell_cluster', 'study_id', 'dataset_id', 'sample_name', 'donor_id', 'sample_type', 'sample_collection_method', 'sample_cell_line_name_cellosaurus', 'sample_cell_line_accession_cellosaurus', 'sample_tissue', 'sample_tissue_uberon_name', 'sample_tissue_uberon_id', 'sample_pathology', 'sample_disease', 'sample_disease_id', 'sample_in_vivo_harvest_timepoint', 'sample_treatment', 'sample_treatment_substance', 'sample_treatment_substance_id', 'sample_treatment_dose', 'sample_treatment_dose_unit', 'sample_treatment_timepoint', 'sample_treatment_timepoint_unit', 'sample_biomarkers', 'sam